# C2 · Wrong labels: where k = 1 collapses and large k holds up

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Corrupt a growing share of the training labels (0%, 10%, 20%, 30%, 40%) and measure, for each level, the test accuracy as a function of `k`: the KNN with `k = 1` **memorizes** the errors, while the best `k` grows with the noise.
Check that cross-validation on the noisy training set still finds a `k` that works.

## Dataset

`make_moons(2000, noise=0.15)` split 50/50: `X_tr, X_te, y_tr, y_te` (the test set has **clean labels**). `noisy(p)` (given) returns the training labels with a fraction `p` swapped at random.

## Educational purpose

Understand KNN as a **local average model**: with `k = 1` it copies every error and its accuracy collapses with the noise; with large `k` it votes among many neighbors and the errors compensate. Noise in the labels is the situation in which the
bias-variance tradeoff is most evident.

## Task

1. For `p` in `[0, 0.1, 0.2, 0.3, 0.4]` and `k` in `Ks = [1, 3, 5, 9, 15, 25, 41, 61, 91, 121]`: train `KNeighborsClassifier(k)` on the training set with labels `noisy(p)` and measure the accuracy on the **clean** test set. Save `acc` (DataFrame rows `p`, columns `k`).
2. Save `optimal_k` (`pd.Series` indexed by `p`: the `k` with maximum test accuracy), `optimal_acc` (the corresponding accuracy) and `k1_acc` (accuracy with `k = 1`).
3. Save `k1_train_acc` (`pd.Series`): training accuracy of `k = 1` **measured on the noisy labels** (the ones it was trained with). What is it? What sense does it make as a measure?
4. **Without looking at the test set:** for each `p` choose `k` with `GridSearchCV` (5 folds) on the noisy training set: save `cv_k` (`pd.Series`) and `acc_with_cv_k` (test accuracy with that `k`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = make_moons(2000, noise=0.15, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
Ks = [1, 3, 5, 9, 15, 25, 41, 61, 91, 121]
ps = [0, 0.1, 0.2, 0.3, 0.4]


def noisy(p):
    """Training labels with a fraction p swapped at random (the test set stays clean)."""
    swap = np.random.default_rng(0).random(len(y_tr)) < p
    return np.where(swap, 1 - y_tr, y_tr)

## Your work

In [ ]:
acc = pd.DataFrame(index=ps, columns=Ks, dtype=float)
for p in ps:
    for k in Ks:
        ...
acc.round(3)

In [ ]:
optimal_k = None
optimal_acc = None
k1_acc = None
k1_train_acc = None

In [ ]:
cv_k = None
acc_with_cv_k = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`KNeighborsClassifier(k).fit(X_tr, noisy(p)).score(X_te, y_te)`. `optimal_k = acc.idxmax(axis=1)`; `optimal_acc = acc.max(axis=1)`; for the training set: `m.score(X_tr, noisy(p))`. `GridSearchCV(KNeighborsClassifier(), {"n_neighbors": Ks}, cv=5).fit(X_tr, noisy(p))` and then `best_params_["n_neighbors"]`.'):
    check(acc.shape == (5, 10) and acc.notna().all().all(), "complete 5 x 10 table", "one accuracy for every pair (p, k)")
    check(optimal_k.is_monotonic_increasing and optimal_k[0] <= 9 and optimal_k[0.4] >= 61, "the optimal k grows with the noise: from 9 or less (p = 0) to 61 or more (p = 0.4)", "argmax per row of acc")
    check(optimal_acc.min() > 0.97, "with the right k the accuracy stays above 0.97 even with 40% wrong labels", "optimal_acc = maximum per row")
    check(k1_acc[0.4] < 0.7 and k1_acc[0] > 0.97 and k1_acc[0.4] < k1_acc[0.2] < k1_acc[0.1] < k1_acc[0], "with k = 1 the accuracy keeps falling, down to below 0.7", "the k = 1 column of acc")
    check((k1_train_acc == 1.0).all(), "with k = 1 the training accuracy (on noisy labels) is always 1.0: it memorizes the errors too", "the nearest neighbor of a training point is the point itself")
    check(cv_k[0.3] >= 25 and cv_k[0.4] >= 25 and (acc_with_cv_k > 0.97).all(), "cross-validation on the noisy training set finds large k (25 or more with 30-40% noise) and the test accuracy stays above 0.97", "GridSearchCV(KNeighborsClassifier(), {'n_neighbors': Ks}, cv=5)")
done()

---
The commented solution is in `solutions/C_break_the_model/C2_noisy_labels_and_choice_of_k.ipynb`. Open it after you pass the check, or after you have really tried.